# The hierarchical organization of urban social overlap

Code for the figures and tables of

> B. Poudyal, M. Macedo, R. Menezes and G. Ghoshal, *The hierarchical organization of urban social overlap: How shared social spaces diverge from physical mobility*, [arXiv:2609.12888](https://arxiv.org/abs/2609.12888) (2026).

| Section | Output |
|---|---|
| 3 | Fig. 3(a, b): distributions of colocation and co-connectedness |
| 4 | Fig. 3(c, d): LouBar hotspots and KDE functional centres |
| 5 | Fig. 4: interaction matrix and its two null models |
| 6 | Tables I and II (`Results/city_metrics.csv`) |
| 7 | Fig. 5: mobility and social hierarchy |
| 8 | Fig. 6: catchment efficiency against social hierarchy |
| 9 | Supplementary Fig. 5: Node and Edge Shuffling in every city |
| 10 | Supplementary Figs. 6–8: colocation and mobility centres, level-1 and level-2 locations |

Sections 1 and 2 hold the configuration and the shared functions and must be run first. After that the sections are independent, except that sections 7–9 read the table written by section 6.

Figs. 1 and 2 are schematics. Supplementary Fig. 3 is section 3 run on the 15- and 45-minute data (change `DATA_DIR`). The mobility hierarchy of Fig. 5 is computed from the raw call records by `mobility_hierarchy.py`.

## 1. Configuration

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path("Data")        # 3_unified_social_metrics_<City>.csv, written by colocation_coconnectedness.py
MOBILITY_DIR = DATA_DIR / "Mobility_hotspots"       # hotspots_loubar_<City>.csv (antenna, activity_volume)
MOBILITY_PHI_DIR = DATA_DIR / "Mobility_hierarchy"  # hierarchical_flow_<City>.txt, written by mobility_hierarchy.py
WORLDPOP_RASTER = DATA_DIR / "bra-ppp-2016-UNadj.tif"
FIGURE_DIR = Path("Figures")
RESULTS_DIR = Path("Results")

# The order fixes the colour of each city in Fig. 3(a, b).
CITIES = [
    "Belem", "Campinas", "Fortaleza", "Guarulhos", "Maceio", "Manaus", "Recife",
    "Salvador", "Sao_Luis", "Sao_Paulo", "Belo_Horizonte", "Goiania", "Brasilia",
]
EXAMPLE_CITY = "Brasilia"      # city shown in Figs. 3(c, d) and 4

# Fall back on any GeoTIFF in DATA_DIR if the raster is stored under another name.
if not WORLDPOP_RASTER.exists():
    rasters = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in (".tif", ".tiff"))
    if rasters:
        WORLDPOP_RASTER = rasters[0]
print("WorldPop raster:", WORLDPOP_RASTER if WORLDPOP_RASTER.exists() else "not found")

# The maps of the paper use the CARTO Positron basemap, which needs a (free) API key from
# carto.com/basemaps/apikey. Set it in the environment, e.g. `export CARTO_API_KEY=...`;
# without a key the maps are drawn on the Esri light-gray basemap.
CARTO_API_KEY = os.environ.get("CARTO_API_KEY", "")

N_NULL = 100                   # realisations of each null model
N_BOOT = 100                   # bootstrap resamples in Fig. 3(a, b)
SEED = 42

## 2. Shared functions

In [ ]:
import re
import unicodedata

import contextily as cx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
from scipy import stats
from scipy.ndimage import maximum_filter
from scipy.spatial import KDTree
from scipy.spatial.distance import cdist
from scipy.stats import gaussian_kde

### Data

`3_unified_social_metrics_<City>.csv` has the columns `metric_type, antenna1, lat1, long1, antenna2, lat2, long2, value`. Rows with `metric_type = colocation` hold the colocation $C_A$ of `antenna1` (one row per antenna); rows with `metric_type = co-connectedness` hold $O_{AB}$ of the pair (`antenna1`, `antenna2`) (one row per pair).

In [ ]:
def label(city):
    """Name of a city as printed in the figures."""
    return city.replace("_", " ")


def _key(name):
    """Lower-case ASCII letters and digits of a name: 'São Luís' -> 'saoluis'."""
    plain = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", plain.lower())


def find_file(directory, prefix, city, suffix=".csv"):
    """Path of <prefix><city><suffix>; the city is matched ignoring case, accents and separators."""
    for path in sorted(Path(directory).glob(f"*{prefix}*{suffix}")):
        if _key(path.stem.split(prefix, 1)[1]) == _key(city):
            return path
    raise FileNotFoundError(f"No '{prefix}<city>{suffix}' file for {city!r} in {directory}")


def load_city(city):
    """Return (antennas, edges) of one city.

    antennas: one row per antenna, indexed by antenna id, with lat, lon and colocation.
    edges: antenna1, antenna2, weight (co-connectedness); every pair is listed once.
    """
    raw = pd.read_csv(find_file(DATA_DIR, "unified_social_metrics_", city))
    antennas = (
        raw[raw["metric_type"] == "colocation"]
        .astype({"antenna1": "int64"})
        .groupby("antenna1")
        .agg(lat=("lat1", "first"), lon=("long1", "first"), colocation=("value", "sum"))
        .rename_axis("antenna")
    )
    edges = (
        raw.loc[raw["metric_type"] == "co-connectedness", ["antenna1", "antenna2", "value"]]
        .dropna()
        .astype({"antenna1": "int64", "antenna2": "int64"})
        .rename(columns={"value": "weight"})
        .reset_index(drop=True)
    )
    return antennas, edges


def load_activity(city, kind="colocation"):
    """Antennas with coordinates and a positive weight: 'colocation' or 'mobility' activity."""
    antennas = load_city(city)[0].dropna(subset=["lat", "lon"])
    if kind == "colocation":
        weight = antennas["colocation"]
    else:
        mobility = pd.read_csv(find_file(MOBILITY_DIR, "hotspots_loubar_", city))
        weight = mobility.set_index("antenna")["activity_volume"].reindex(antennas.index)
    antennas = antennas.assign(weight=weight)
    return antennas[antennas["weight"] > 0]


def read_mobility_phi(city):
    """Phi of the mobility network of a city (written by mobility_hierarchy.py); NaN if missing."""
    try:
        return float(find_file(MOBILITY_PHI_DIR, "hierarchical_flow_", city, ".txt").read_text())
    except FileNotFoundError:
        return np.nan

### LouBar levels

The tangent to the Lorenz curve at its upper end has slope $S = \max/\mathrm{mean}$ and meets the horizontal axis at $1 - 1/S$; entries ranked above that point are the hotspots (Louail *et al.*, 2014). Iterating on the remaining entries gives the activity levels (Supplementary Note 4).

In [ ]:
def loubar_cut(ascending):
    """Index of the LouBar cut in an ascending array of positive values."""
    n = len(ascending)
    slope = ascending[-1] / np.mean(ascending)
    return min(int((1.0 - 1.0 / slope) * n), n - 1)


def loubar_threshold(values):
    """Value at the LouBar cut of the positive entries of an array."""
    values = np.asarray(values).ravel()
    values = np.sort(values[values > 0])
    return values[loubar_cut(values)] if len(values) else 0.0


def loubar_levels(values, min_size=5):
    """Activity level of every entry with a positive value (1 = most active).

    The entries above the cut form level 1 and are removed; the cut is recomputed on the
    rest for level 2, and so on. Fewer than `min_size` remaining entries form the last level.
    Equal values keep their order in `values` (stable sort), so a cut inside a group of
    ties gives the same split on every machine.
    """
    values = values[values > 0].sort_values(kind="stable")
    ascending = values.to_numpy()
    levels = np.zeros(len(ascending), dtype=int)
    remaining, level = len(ascending), 1
    while remaining > 0:
        cut = loubar_cut(ascending[:remaining]) if remaining >= min_size else 0
        levels[cut:remaining] = level
        remaining, level = cut, level + 1
    return pd.Series(levels, index=values.index, name="level")

### Hierarchy and null models

$T_{ij}$ is the co-connectedness between antennas of levels $i$ and $j$, as a symmetric matrix. $\Phi$ (Eq. 5) is the share of the matrix on the main diagonal and the two adjacent ones.

- **Node Shuffling** permutes colocation across antennas and recomputes the levels; the network is untouched.
- **Edge Shuffling** redistributes the total overlap over the cells of the matrix with the probabilities of Eq. (S1), $T^h_{ij} \propto \sum_k T_{ik} \sum_m T_{mj}$.

In [ ]:
def interaction_matrix(levels, edges):
    """Co-connectedness summed between activity levels (raw weights, L x L, symmetric)."""
    i = edges["antenna1"].map(levels).to_numpy(dtype=float)
    j = edges["antenna2"].map(levels).to_numpy(dtype=float)
    known = ~(np.isnan(i) | np.isnan(j))  # edges touching an antenna without a level are ignored
    i, j = i[known].astype(int) - 1, j[known].astype(int) - 1
    weight = edges["weight"].to_numpy(dtype=float)[known]

    n = int(levels.max())
    once = np.bincount(i * n + j, weights=weight, minlength=n * n).reshape(n, n)
    return once + once.T - np.diag(once.diagonal())


def phi(matrix):
    """Eq. (5): share of the total on the main diagonal and the two adjacent diagonals."""
    band = np.trace(matrix) + np.trace(matrix, offset=1) + np.trace(matrix, offset=-1)
    return float(band / matrix.sum())


def edge_shuffle_expectation(matrix):
    """Eq. (S1): expected matrix when levels mix at random given their total overlap."""
    return np.outer(matrix.sum(axis=1), matrix.sum(axis=0)) / matrix.sum()


def node_shuffle(colocation, edges, n_iter=N_NULL, seed=SEED):
    """Interaction matrices of the Node Shuffling null model, one per realisation."""
    rng = np.random.RandomState(seed)
    for _ in range(n_iter):
        shuffled = pd.Series(rng.permutation(colocation.to_numpy()), index=colocation.index)
        yield interaction_matrix(loubar_levels(shuffled), edges)


def edge_shuffle_phi(matrix, n_iter=N_NULL, seed=SEED):
    """Phi of `n_iter` Edge Shuffling realisations: multinomial draws of the total overlap."""
    rng = np.random.RandomState(seed)
    expected = edge_shuffle_expectation(matrix)
    draws = rng.multinomial(int(matrix.sum()), (expected / expected.sum()).ravel(), size=n_iter)
    row, column = np.indices(matrix.shape)
    in_band = (np.abs(row - column) <= 1).ravel()
    return draws[:, in_band].sum(axis=1) / draws.sum(axis=1)


def hierarchy_summary(colocation, edges):
    """Phi of a city, with mean, standard deviation and Z-score of both null models."""
    matrix = interaction_matrix(loubar_levels(colocation), edges)
    summary = {"n_levels": len(matrix), "phi": phi(matrix)}
    nulls = {
        "node": np.array([phi(m) for m in node_shuffle(colocation, edges)]),
        "edge": edge_shuffle_phi(matrix),
    }
    for name, values in nulls.items():
        summary[f"phi_{name}_null"] = values.mean()
        summary[f"phi_{name}_null_std"] = values.std()
        summary[f"z_{name}_null"] = (summary["phi"] - values.mean()) / values.std()
    return summary

### Functional centres and catchment efficiency

Antenna activity is smoothed with a weighted Gaussian KDE (Scott's rule) in EPSG:3857, on a grid with about three points per median antenna spacing $d_\mathrm{med}$. A centre is a local maximum of the surface whose density exceeds the LouBar threshold computed over all grid values (Supplementary Note 8).

Residential population is the sum of the WorldPop pixels in the Voronoi cell of each antenna site (Supplementary Note 9). The catchment efficiency $\eta$ follows Eqs. (7)–(9).

In [ ]:
EARTH_RADIUS = 6378137.0  # WGS84 semi-major axis in metres
MAX_GRID = 1500  # upper limit on the number of grid points per axis


def to_web_mercator(lon, lat):
    """Longitude and latitude in degrees (WGS84) to EPSG:3857 metres."""
    lon, lat = np.asarray(lon, dtype=float), np.asarray(lat, dtype=float)
    x = EARTH_RADIUS * np.radians(lon)
    y = EARTH_RADIUS * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))
    return x, y


def antenna_spacing(x, y, fallback=500.0):
    """d_med: median distance from an antenna site to its nearest neighbouring site."""
    sites = np.unique(np.column_stack([x, y]), axis=0)  # antennas sharing coordinates are one site
    if len(sites) < 2:
        return fallback
    nearest = KDTree(sites).query(sites, k=2)[0][:, 1]
    nearest = nearest[nearest > 1.0]
    return float(np.median(nearest)) if len(nearest) else fallback


def find_centers(x, y, weights):
    """Functional centres, shape (k, 2), and the surface (grid_x, grid_y, density) they lie on."""
    spacing = antenna_spacing(x, y)
    width = max(x.max() - x.min(), 1000.0)
    n = max(100, min(int(width * 3 / spacing), MAX_GRID))
    window = max(3, int(spacing / (width / n)))  # neighbourhood of a local maximum, in grid points

    grid_x, grid_y = np.mgrid[x.min():x.max():complex(0, n), y.min():y.max():complex(0, n)]
    kde = gaussian_kde([x, y], weights=weights)
    density = kde(np.vstack([grid_x.ravel(), grid_y.ravel()])).reshape(grid_x.shape)
    density = (density - density.min()) / (density.max() - density.min())

    is_peak = maximum_filter(density, size=window) == density
    is_center = is_peak & (density > loubar_threshold(density))
    return np.column_stack([grid_x[is_center], grid_y[is_center]]), (grid_x, grid_y, density)


def voronoi_population(antennas, raster_path=None, buffer_deg=0.05):
    """WorldPop population in the Voronoi cell of each antenna (needs geopandas, shapely, rasterio).

    The tessellation is built on the antenna sites and extends to the bounding box of their
    convex hull buffered by `buffer_deg` degrees. Antennas sharing a site share a cell, and
    each of them is assigned the population of that cell.
    """
    import geopandas as gpd
    import rasterio
    from rasterio.mask import mask
    from shapely.geometry import MultiPoint
    from shapely.ops import voronoi_diagram

    points = gpd.GeoDataFrame(
        antennas, geometry=gpd.points_from_xy(antennas["lon"], antennas["lat"]), crs="EPSG:4326"
    )
    sites = MultiPoint(points.geometry.tolist())
    cells = voronoi_diagram(sites, envelope=sites.convex_hull.buffer(buffer_deg))
    cells = gpd.GeoDataFrame(geometry=list(cells.geoms), crs=points.crs)
    cell_of_antenna = gpd.sjoin(points, cells, how="left", predicate="within")["index_right"]

    population = {}
    with rasterio.open(raster_path or WORLDPOP_RASTER) as raster:
        for cell_id, cell in cells.geometry.items():
            try:
                pixels = mask(raster, [cell], crop=True)[0][0]
            except ValueError:  # raised by rasterio when the cell lies outside the raster
                population[cell_id] = 0.0
                continue
            valid = pixels > 0
            if raster.nodata is not None:
                valid &= pixels != raster.nodata
            population[cell_id] = float(pixels[valid].sum())
    return cell_of_antenna.map(population).fillna(0.0).rename("population")


def catchment(city):
    """Social centres, residential centres and catchment efficiency eta (%) of a city.

    All three use the antennas with positive colocation: the social centres are the KDE
    centres of their colocation, the residential centres those of the population of their
    Voronoi cells. eta = 100 (D_res - D_col) / D_res, where D is the colocation-weighted
    distance from the antennas to the nearest centre of each kind.
    """
    active = load_activity(city, "colocation")
    x, y = to_web_mercator(active["lon"], active["lat"])
    colocation = active["weight"].to_numpy()
    population = voronoi_population(active).reindex(active.index)
    social = find_centers(x, y, colocation)[0]
    residential = find_centers(x, y, population.to_numpy())[0]

    antennas = np.column_stack([x, y])
    d_res = np.sum(colocation * cdist(antennas, residential).min(axis=1))
    d_col = np.sum(colocation * cdist(antennas, social).min(axis=1))
    return social, residential, float(100.0 * (d_res - d_col) / d_res)

### Plotting helpers

In [ ]:
CARTO_POSITRON = "https://a.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png?key="


def paper_style(font_scale=1.4):
    """Serif fonts on a white grid."""
    sns.set_theme(context="paper", style="whitegrid", font_scale=font_scale)
    plt.rcParams.update({"font.family": "serif", "axes.edgecolor": "#333333", "axes.linewidth": 0.8})


def add_basemap(ax):
    """Basemap tiles under an axes in EPSG:3857 coordinates."""
    if CARTO_API_KEY:
        credit = "(C) OpenStreetMap contributors (C) CARTO"
        cx.add_basemap(ax, source=CARTO_POSITRON + CARTO_API_KEY, attribution=credit)
    else:
        cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)


def save_figure(fig, name):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=300, bbox_inches="tight")

## 3. Fig. 3(a, b): distributions of colocation and co-connectedness

Points are log-binned densities of the individual cities. The dashed line is a log-normal fit to the pooled data; in (b) the tail is fitted with a power law, whose lower cutoff $x_\mathrm{min}$ minimises the Kolmogorov–Smirnov distance (`powerlaw` package). Bands are 95% bootstrap intervals obtained by resampling cities with replacement.

In [ ]:
import powerlaw

CITY_POINTS = {"s": 50, "alpha": 0.5, "edgecolors": "none", "zorder": 2}


def log_binned_pdf(values, n_edges=25):
    """Probability density of the values in logarithmic bins; empty bins are dropped."""
    edges = np.logspace(np.log10(values.min()), np.log10(values.max()), n_edges)
    counts, _ = np.histogram(values, bins=edges)
    filled = counts > 0
    centers = np.sqrt(edges[:-1] * edges[1:])
    return centers[filled], (counts / counts.sum() / np.diff(edges))[filled]


def lognormal_fit(values):
    """Maximum-likelihood (sigma, mu) of a log-normal distribution."""
    sigma, _, scale = stats.lognorm.fit(values, floc=0)
    return sigma, np.log(scale)


def lognormal_pdf(x, sigma, mu):
    return stats.lognorm.pdf(x, sigma, loc=0, scale=np.exp(mu))


def tail_exponent(values, xmin):
    """Maximum-likelihood exponent of a continuous power law above xmin."""
    tail = values[values >= xmin]
    return 1 + len(tail) / np.log(tail / xmin).sum()


def tail_pdf(x, values, xmin, alpha):
    """Power-law density above xmin, scaled by the share of the values in the tail."""
    return np.mean(values >= xmin) * (alpha - 1) / xmin * (x / xmin) ** -alpha


def bootstrap_band(samples, curve, rng, level=0.95):
    """Lower and upper percentile of curve(pooled values) when cities are resampled."""
    cities = list(samples)
    curves = [
        curve(np.concatenate([samples[city] for city in rng.choice(cities, size=len(cities))]))
        for _ in range(N_BOOT)
    ]
    return np.percentile(curves, [50 * (1 - level), 50 * (1 + level)], axis=0)


def plot_distribution(ax, samples, xlabel, rng, tail=False):
    """One panel of Fig. 3(a, b); returns the scatter handles of the cities."""
    pooled = np.concatenate(list(samples.values()))
    x = np.logspace(np.log10(pooled.min()), np.log10(pooled.max()), 200)

    cities = []
    colors = plt.cm.plasma(np.linspace(0, 1, len(samples)))
    for (city, values), color in zip(samples.items(), colors):
        if len(values) >= 10 and values.min() < values.max():
            binned = log_binned_pdf(values)
            cities.append(ax.scatter(*binned, color=color, label=label(city), **CITY_POINTS))

    sigma, mu = lognormal_fit(pooled)
    band = bootstrap_band(samples, lambda v: lognormal_pdf(x, *lognormal_fit(v)), rng)
    name = "Log-Normal Body\n(Fits the head)" if tail else f"Global Log-Normal Fit\n(σ={sigma:.2f}, μ={mu:.2f})"
    fits = [ax.fill_between(x, *band, color="gray", alpha=0.3, label="95% CI (Log-Normal)", zorder=1)]
    fits += ax.plot(x, lognormal_pdf(x, sigma, mu), "k--", linewidth=3 if tail else 4, label=name, zorder=10)
    print(f"{xlabel}: log-normal sigma = {sigma:.2f}, mu = {mu:.2f}")

    if tail:
        fit = powerlaw.Fit(pooled, xmin=None, verbose=False)
        xmin, alpha = fit.xmin, fit.power_law.alpha
        x = x[x >= xmin]
        band = bootstrap_band(samples, lambda v: tail_pdf(x, v, xmin, tail_exponent(v, xmin)), rng)
        name = "Power Law Tail\n" + rf"($\alpha={alpha:.2f}, x_{{min}}={xmin:.0f}$)"
        fits.append(ax.fill_between(x, *band, color="red", alpha=0.2, label="95% CI (Power Law Tail)", zorder=2))
        fits += ax.plot(x, tail_pdf(x, pooled, xmin, alpha), "r-", linewidth=4, label=name, zorder=11)
        fits.append(ax.axvline(xmin, color="gray", linestyle=":", linewidth=2, label="Tail Threshold"))
        print(f"{xlabel}: power-law tail alpha = {alpha:.2f}, x_min = {xmin:.0f}")

    ax.set(xscale="log", yscale="log")
    ax.set_xlabel(xlabel, fontsize=20, fontweight="bold", labelpad=10)
    ax.set_ylabel("Probability Density", fontsize=20, fontweight="bold", labelpad=10)
    ax.tick_params(axis="both", which="major", labelsize=16)
    ax.legend(handles=fits, loc="lower left" if tail else "upper right", fontsize=13)
    return cities

In [ ]:
colocation, coconnectedness = {}, {}
for city in CITIES:
    antennas, edges = load_city(city)
    colocation[city] = antennas["colocation"][antennas["colocation"] > 0].to_numpy(dtype=float)
    coconnectedness[city] = edges["weight"][edges["weight"] > 0].to_numpy(dtype=float)

paper_style(font_scale=1.6)
rng = np.random.RandomState(SEED)
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(22, 8.5), gridspec_kw={"wspace": 0.25})
plot_distribution(ax_a, colocation, "Colocation Count", rng)
city_handles = plot_distribution(ax_b, coconnectedness, "Co-Connectedness Count", rng, tail=True)
for ax, letter in zip((ax_a, ax_b), "ab"):
    ax.set_title(f"({letter})", loc="left", fontsize=24, fontweight="bold")
fig.legend(handles=city_handles, loc="center left", bbox_to_anchor=(0.91, 0.5), fontsize=14, markerscale=2)
save_figure(fig, "fig3ab_distributions")

## 4. Fig. 3(c, d): hotspots and functional centres

(c) Antennas in LouBar levels 1 and 2 of colocation, with the Lorenz-curve construction of the two cuts. (d) KDE surface of colocation and the functional centres detected on it.

In [ ]:
HUB_MARKERS = {
    1: {"marker": "*", "s": 350, "color": "#e74c3c", "edgecolor": "black", "linewidth": 1},
    2: {"marker": "o", "s": 90, "color": "#3498db", "edgecolor": "white", "linewidth": 0.5},
}
CENTER_MARKER = {"marker": "*", "markersize": 24, "color": "blue", "markeredgecolor": "white", "markeredgewidth": 1.5}
# kind of activity -> (hub name, y label of the Lorenz inset, colour-bar label)
MAP_NAMES = {
    "colocation": ("Hubs", "Cumul. Active Density", "Relative Colocation Density (KDE)"),
    "mobility": ("Mobility Hubs", "Cumul. Mobility Activity", "Relative Mobility Density (KDE)"),
}


def plot_hubs(ax, x, y, levels, hub_name):
    """Antennas of activity level 2 and, drawn over them, of level 1."""
    for level in (2, 1):
        here = levels == level
        ax.scatter(x[here], y[here], alpha=0.9, label=f"Level {level} {hub_name}", **HUB_MARKERS[level])
    ax.legend(loc="best", fontsize=20, frameon=True, framealpha=0.9)


def plot_lorenz_inset(ax, weights, levels, ylabel):
    """Lorenz curve of the weights with the tangents that define the level-1 and level-2 cuts."""
    ascending = np.sort(weights)
    n = len(ascending)
    rank = np.arange(1, n + 1) / n
    share = np.cumsum(ascending) / ascending.sum()
    first_1 = n - np.sum(levels == 1)  # position of the lowest level-1 antenna
    first_2 = first_1 - np.sum(levels == 2)

    backing = Rectangle((0.50, 0.60), 0.50, 0.40, transform=ax.transAxes, color="white", alpha=0.85)
    ax.add_patch(backing).set_zorder(4)
    inset = ax.inset_axes((0.64, 0.70, 0.33, 0.27), zorder=5)
    inset.plot(rank, share, color="black", linewidth=2)
    inset.plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=0.8)
    # A level starts where the tangent at the upper end of the remaining curve meets the
    # axis: the end is (1, 1) for level 1 and the lowest level-1 antenna for level 2.
    for level, first, end, shade in [(1, first_1, n - 1, 0.3), (2, first_2, first_1, 0.2)]:
        color = HUB_MARKERS[level]["color"]
        name = f"L{level} Cut: {rank[first]:.2f}"
        inset.plot([rank[first], rank[end]], [0, share[end]], "--", color=color, label=name)
        inset.fill_between(rank[first:end + 1], share[first:end + 1], color=color, alpha=shade)
        inset.scatter(rank[first], 0, color=color, s=40, zorder=5, clip_on=False)
    inset.set(xlim=(0, 1.05), ylim=(0, 1.05))
    inset.set_xlabel("Rank (%)", fontsize=16)
    inset.set_ylabel(ylabel, fontsize=16)
    inset.tick_params(labelsize=12)
    inset.legend(loc="upper left", fontsize=12, framealpha=0.9)


def plot_density(ax, grid_x, grid_y, density, centers, colorbar_label):
    """KDE surface with the functional centres marked by stars."""
    surface = ax.contourf(grid_x, grid_y, density, levels=50, cmap="Reds", alpha=0.6)
    ax.plot(centers[:, 0], centers[:, 1], linestyle="none", zorder=10, **CENTER_MARKER)
    colorbar_ax = ax.inset_axes((0.1, -0.07, 0.8, 0.03))
    colorbar = ax.figure.colorbar(surface, cax=colorbar_ax, orientation="horizontal")
    colorbar.set_label(colorbar_label, fontsize=20, fontweight="bold", labelpad=10)
    colorbar.ax.tick_params(labelsize=16)


def plot_hubs_and_centers(ax_hubs, ax_centers, city, kind="colocation", letters="cd"):
    """Hotspot map and KDE-centre map of one kind of activity; returns the number of centres."""
    hub_name, lorenz_label, colorbar_label = MAP_NAMES[kind]
    activity = load_activity(city, kind)
    x, y = to_web_mercator(activity["lon"], activity["lat"])
    levels = loubar_levels(activity["weight"]).reindex(activity.index).to_numpy()
    centers, surface = find_centers(x, y, activity["weight"].to_numpy())

    plot_hubs(ax_hubs, x, y, levels, hub_name)
    plot_density(ax_centers, *surface, centers, colorbar_label)
    # Both maps show the area spanned by the hubs and the centres, with a 5% margin.
    shown = np.vstack([np.column_stack([x, y])[levels <= 2], centers])
    low, high = shown.min(axis=0), shown.max(axis=0)
    margin = 0.05 * (high - low)
    for ax, letter in zip((ax_hubs, ax_centers), letters):
        ax.set(xlim=(low[0] - margin[0], high[0] + margin[0]), ylim=(low[1] - margin[1], high[1] + margin[1]))
        ax.set_aspect("equal")
        add_basemap(ax)
        ax.set_axis_off()
        ax.set_title(f"({letter})", loc="left", fontsize=24, fontweight="bold")
    plot_lorenz_inset(ax_hubs, activity["weight"].to_numpy(), levels, lorenz_label)
    return len(centers)

In [ ]:
plt.rcdefaults()
fig, axes = plt.subplots(1, 2, figsize=(24, 12), gridspec_kw={"wspace": 0.05})
n_centers = plot_hubs_and_centers(*axes, EXAMPLE_CITY)
print(f"{label(EXAMPLE_CITY)}: {n_centers} social centre(s)")
save_figure(fig, f"fig3cd_centers_{EXAMPLE_CITY}")

## 5. Fig. 4: interaction matrix and null models

(a) Empirical matrix, (b) one Node Shuffling realisation, (c) Edge Shuffling expectation, on a common colour scale.

In [ ]:
FLOOR = 1e-7  # added to every cell so that empty cells can be drawn on the logarithmic colour scale


def plot_matrices(matrices, titles):
    """Three heat maps on one colour scale: two on top, the third centred below."""
    shares = [m / m.sum() for m in matrices]
    norm = LogNorm(vmin=min(s[s > 0].min() for s in shares), vmax=max(s.max() for s in shares))
    level_names = list(range(1, len(shares[0]) + 1))

    fig = plt.figure(figsize=(15, 12))
    grid = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.4)
    axes = [fig.add_subplot(grid[0, 0:2]), fig.add_subplot(grid[0, 2:4]), fig.add_subplot(grid[1, 1:3])]
    colorbar_ax = axes[-1].inset_axes((1.08, 0.0, 0.05, 1.0))
    for ax, share, title, letter in zip(axes, shares, titles, "abc"):
        last = ax is axes[-1]
        sns.heatmap(share + FLOOR, ax=ax, cmap="Blues", norm=norm, xticklabels=level_names,
                    yticklabels=level_names, cbar=last, cbar_ax=colorbar_ax if last else None)
        ax.set_title(title, fontsize=20, fontweight="bold")
        ax.set_title(f"({letter})\n", loc="left", fontsize=20, fontweight="bold")
        ax.set_xlabel("Hotspot Level", fontsize=16)
        ax.set_ylabel("Hotspot Level", fontsize=16)
        ax.tick_params(labelsize=12)
    colorbar_ax.set_ylabel("Normalized Flow (Log Scale)", fontsize=14, fontweight="bold")
    return fig

In [ ]:
antennas, edges = load_city(EXAMPLE_CITY)
empirical = interaction_matrix(loubar_levels(antennas["colocation"]), edges)
node_null = next(node_shuffle(antennas["colocation"], edges, n_iter=1))
edge_null = edge_shuffle_expectation(empirical)

paper_style(font_scale=1.4)
fig = plot_matrices(
    [empirical, node_null, edge_null],
    [
        f"Empirical {label(EXAMPLE_CITY)}\n" + rf"$\Phi$ = {phi(empirical):.3f}",
        "Node Shuffling Null\n" + rf"$\Phi_{{node}}$ = {phi(node_null):.3f}",
        "Edge Shuffling Null\n" + rf"$\Phi_{{edge}}$ = {phi(edge_null):.3f}",
    ],
)
save_figure(fig, f"fig4_interaction_matrices_{EXAMPLE_CITY}")

## 6. Tables I and II: metrics of every city

One row per city, saved to `Results/city_metrics.csv` and used by sections 7–9. Columns whose input is missing are left empty:

- `n_mobility_centers` needs `Mobility_hotspots/hotspots_loubar_<City>.csv`;
- `phi_mobility` needs `Mobility_hierarchy/hierarchical_flow_<City>.txt` (written by `mobility_hierarchy.py`);
- `n_residential_centers` and `catchment_efficiency` need the WorldPop raster.

In [ ]:
def city_metrics(city):
    antennas, edges = load_city(city)
    row = {"city": city, "n_nodes": pd.unique(edges[["antenna1", "antenna2"]].to_numpy().ravel()).size}
    row.update(n_edges=len(edges), **hierarchy_summary(antennas["colocation"], edges))
    row["phi_mobility"] = read_mobility_phi(city)

    try:
        mobility = load_activity(city, "mobility")
        x, y = to_web_mercator(mobility["lon"], mobility["lat"])
        row["n_mobility_centers"] = len(find_centers(x, y, mobility["weight"].to_numpy())[0])
    except FileNotFoundError:
        row["n_mobility_centers"] = np.nan

    if WORLDPOP_RASTER.exists():
        social, residential, eta = catchment(city)
        row.update(n_residential_centers=len(residential), catchment_efficiency=eta)
    else:
        active = load_activity(city, "colocation")
        social = find_centers(*to_web_mercator(active["lon"], active["lat"]), active["weight"].to_numpy())[0]
        row.update(n_residential_centers=np.nan, catchment_efficiency=np.nan)
    row["n_social_centers"] = len(social)
    return row


metrics = pd.DataFrame([city_metrics(city) for city in CITIES])
if not WORLDPOP_RASTER.exists():
    print(f"No population raster in {DATA_DIR}: residential centres and eta are left empty.")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(RESULTS_DIR / "city_metrics.csv", index=False)

In [ ]:
def centricity(n_centers):
    if pd.isna(n_centers):
        return "-"
    return {1: "Monocentric", 2: "Bi-centric"}.get(int(n_centers), "Polycentric")


def as_text(column, spec):
    """Format a column for printing, with '-' for missing values."""
    return column.map(lambda value: "-" if pd.isna(value) else format(value, spec))


named = metrics.assign(city=metrics["city"].map(label))

rows = named.sort_values("n_social_centers", ascending=False, kind="stable")
print("Table I. Mobility and social centricity")
print(pd.DataFrame({
    "City": rows["city"],
    "N_mob": as_text(rows["n_mobility_centers"], ".0f"),
    "Mobility centricity": rows["n_mobility_centers"].map(centricity),
    "N_col": as_text(rows["n_social_centers"], ".0f"),
    "Social centricity": rows["n_social_centers"].map(centricity),
}).to_string(index=False))

rows = named.sort_values("catchment_efficiency", ascending=False, kind="stable")
print("\nTable II. Catchment efficiency and social hierarchy")
print(pd.DataFrame({
    "City": rows["city"],
    "N_res": as_text(rows["n_residential_centers"], ".0f"),
    "N_col": as_text(rows["n_social_centers"], ".0f"),
    "eta (%)": as_text(rows["catchment_efficiency"], "+.2f"),
    "Phi": as_text(rows["phi"], ".3f"),
}).to_string(index=False))

## 7. Fig. 5: mobility and social hierarchy

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / "city_metrics.csv")
table = metrics.assign(gap=metrics["phi_mobility"] - metrics["phi"]).sort_values("gap")
position = np.arange(len(table))

paper_style(font_scale=1.4)
layout = {"height_ratios": [2, 1.2], "hspace": 0.15}
fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw=layout)

top.vlines(position, table["phi"], table["phi_mobility"], color="gray", alpha=0.5, linewidth=2, zorder=1)
top.scatter(position, table["phi_mobility"], s=70, color="#e67e22", label="Mobility Hierarchy (Individual)", zorder=3)
top.scatter(position, table["phi"], s=70, color="#2c3e50", label="Social Hierarchy (Pair)", zorder=3)
top.set_ylabel(r"Flow Hierarchy ($\Phi$)", fontsize=16, fontweight="bold")
top.margins(y=0.2)
top.legend(loc="lower left", frameon=True, framealpha=0.9, fontsize=12)

bars = bottom.bar(position, table["gap"], color="#7f8c8d", alpha=0.85, width=0.6)
bottom.bar_label(bars, fmt="%.2f", padding=3, fontsize=11, fontweight="bold")
bottom.set_title(r"Gap ($\Phi_{Mobility} - \Phi_{Social}$)", fontsize=16, pad=10)
bottom.set_ylabel(r"Gap ($\Delta\Phi$)", fontsize=16, fontweight="bold")
bottom.margins(y=0.15)
bottom.grid(axis="x", alpha=0)
bottom.set_xticks(position, table["city"].map(label), rotation=45, ha="right", fontsize=14)
for ax in (top, bottom):
    ax.tick_params(axis="y", labelsize=12)
save_figure(fig, "fig5_hierarchy_gap")

## 8. Fig. 6: catchment efficiency against social hierarchy

In [ ]:
# Offset (in eta, in Phi) of each city name from its bubble.
LABEL_OFFSETS = {
    "Sao_Paulo": (0, 0.006), "Brasilia": (0, -0.007), "Campinas": (0, -0.006),
    "Belo_Horizonte": (1.7, 0.005), "Fortaleza": (0, 0.005), "Salvador": (0, 0.005),
    "Belem": (0, -0.006), "Maceio": (0, -0.005), "Sao_Luis": (0, 0.005), "Recife": (-2, 0.004),
    "Manaus": (-1, -0.005), "Goiania": (0, 0.005), "Guarulhos": (0, -0.006),
}
LABEL_STYLE = {
    "ha": "center", "va": "center", "color": "#2f3640", "fontsize": 18, "fontweight": "bold",
    "bbox": {"facecolor": "white", "alpha": 0.6, "edgecolor": "none", "pad": 1},
}

metrics = pd.read_csv(RESULTS_DIR / "city_metrics.csv")
eta, social_phi = metrics["catchment_efficiency"], metrics["phi"]
assert eta.notna().all(), "catchment_efficiency is empty: section 6 needs the WorldPop raster"

sns.set_theme(style="white", context="paper")
fig, ax = plt.subplots(figsize=(15, 11))
ax.axvline(0, color="#7f8fa6", linestyle="--", linewidth=3, zorder=1)
sns.scatterplot(
    data=metrics.astype({"n_social_centers": int}), x="catchment_efficiency", y="phi", ax=ax,
    size="n_social_centers", hue="n_social_centers", sizes=(150, 800), palette="magma_r",
    alpha=0.85, edgecolor="black", linewidth=1.5, zorder=3,
)
for city, x, y in zip(metrics["city"], eta, social_phi):
    dx, dy = LABEL_OFFSETS.get(city, (0, -0.005))
    ax.text(x + dx, y + dy, label(city), zorder=4, **LABEL_STYLE)

ax.set_xlabel(r"Catchment Efficiency ($\eta$ (%))", fontsize=24, fontweight="bold")
ax.set_ylabel(r"Social Hierarchy ($\Phi$)", fontsize=24, fontweight="bold")
ax.tick_params(axis="both", which="major", direction="in", labelsize=18)
ax.set_xlim(min(-30, eta.min() - 10), max(70, eta.max() + 10))
ax.set_ylim(min(0.69, social_phi.min() - 0.005), max(0.82, social_phi.max() + 0.005))
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(title="Functional Centers (N)", title_fontsize=20, fontsize=18, loc="upper left", framealpha=0.9)
fig.tight_layout()
save_figure(fig, "fig6_catchment_vs_hierarchy")

## 9. Supplementary Fig. 5: hierarchy under the two null models

Top: empirical $\Phi$ and the null ensemble (mean with a 95% interval, $1.96\,\sigma$); a star marks $|Z| > 1.96$. Bottom: Z-score of the empirical value against the ensemble.

In [ ]:
def plot_null_comparison(table, null, title):
    """Empirical Phi against one null model ('node' or 'edge') in every city, with Z-scores."""
    table = table.sort_values("phi", ascending=False)
    position = np.arange(len(table))
    mean, std, z = table[f"phi_{null}_null"], table[f"phi_{null}_null_std"], table[f"z_{null}_null"]

    layout = {"height_ratios": [2, 1.2], "hspace": 0.15}
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw=layout)

    top.vlines(position, mean, table["phi"], color="gray", alpha=0.4, linewidth=2, zorder=1)
    top.errorbar(position, mean, yerr=1.96 * std, fmt="o", color="#e74c3c", elinewidth=2, capsize=4,
                 markersize=8, alpha=0.9, label="Null Model (Random)", zorder=2)
    top.scatter(position, table["phi"], s=120, color="#2c3e50", edgecolors="white", linewidth=1.5,
                label="Empirical Data", zorder=3)
    significant = (np.abs(z) > 1.96).to_numpy()
    for x, y in zip(position[significant], table["phi"][significant]):
        top.text(x, y + 0.02, "*", ha="center", va="center", fontsize=18, color="#2c3e50", fontweight="bold")
    top.set_ylabel(r"Flow Hierarchy ($\Phi$)", fontsize=16, fontweight="bold")
    top.set_title(title, fontsize=18, pad=15, fontweight="bold")
    top.set_ylim(0, 1.05)
    top.legend(loc="lower right", frameon=True, framealpha=0.9, fontsize=12)

    bars = bottom.bar(position, z, color="#27ae60", alpha=0.85, width=0.6)
    bottom.bar_label(bars, fmt="%.1f", padding=3, fontsize=11, fontweight="bold")
    bottom.axhline(1.96, color="blue", linestyle="--", linewidth=3)
    bottom.text(len(table) - 0.5, 1.96, "95% Sig. ($Z=1.96$)", color="blue", fontsize=14, va="bottom", ha="right")
    bottom.set_ylabel(r"Z-Score ($\sigma$)", fontsize=16, fontweight="bold")
    bottom.set_title("Statistical Significance of Hierarchy", fontsize=16, pad=10)
    bottom.margins(y=0.15)
    bottom.grid(axis="x", alpha=0)
    bottom.set_xticks(position, table["city"].map(label), rotation=45, ha="right", fontsize=14)
    for ax in (top, bottom):
        ax.tick_params(axis="y", labelsize=12)
    return fig

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / "city_metrics.csv")
paper_style(font_scale=1.4)
for letter, null in zip("ab", ("node", "edge")):
    title = f"({letter}) Empirical vs. {null.capitalize()} Shuffling Null Model"
    save_figure(plot_null_comparison(metrics, null, title), f"figS5{letter}_{null}_shuffling")

## 10. Supplementary Figs. 6–8: colocation and mobility centres

Supplementary Figs. 6 and 7 repeat the two panels of Fig. 3(c, d) for colocation (top) and mobility activity (bottom) in Brasília and São Paulo. Supplementary Fig. 8 shows the level-1 and level-2 colocation antennas of the two cities.

In [ ]:
plt.rcdefaults()
for number, city in ((6, "Brasilia"), (7, "Sao_Paulo")):
    fig, axes = plt.subplots(2, 2, figsize=(24, 24), gridspec_kw={"wspace": 0.05, "hspace": 0.25})
    n_social = plot_hubs_and_centers(*axes[0], city, "colocation", letters="ab")
    n_mobility = plot_hubs_and_centers(*axes[1], city, "mobility", letters="cd")
    print(f"{label(city)}: {n_social} social and {n_mobility} mobility centre(s)")
    save_figure(fig, f"figS{number}_colocation_vs_mobility_centers_{city}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 12), gridspec_kw={"wspace": 0.05})
for ax, city, letter in zip(axes, ("Brasilia", "Sao_Paulo"), "ab"):
    activity = load_activity(city, "colocation")
    x, y = to_web_mercator(activity["lon"], activity["lat"])
    levels = loubar_levels(activity["weight"]).reindex(activity.index).to_numpy()
    plot_hubs(ax, x[levels <= 2], y[levels <= 2], levels[levels <= 2], "Hubs")
    ax.set_aspect("equal")
    add_basemap(ax)
    ax.set_axis_off()
    ax.set_title(f"({letter}) {label(city)}", loc="left", fontsize=24, fontweight="bold")
save_figure(fig, "figS8_level_1_2_locations")